In [4]:
import os
# from getpass import getpass


In [5]:
# os.environ["GROQ_API_KEY"]=getpass("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")

In [12]:
%pip install -q langchain langchain-community langchain-huggingface langchain-groq faiss-cpu pypdf sentence-transformers

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough



[link text](https://)# PDF load

```
# This is formatted as code
```



In [15]:
pdf_path="genai-principles.pdf"

In [17]:
loader=PyPDFLoader(pdf_path)
doc=loader.load()


In [18]:
print("Total Pages:",len(doc))

Total Pages: 12


In [19]:
text=doc[0].page_content


In [20]:
print("total Characters:",len(text))

total Characters: 1506


In [21]:
print(doc[0].page_content[:30000])

Karan Singh, Assistant Professor of Operations Research 
Principles of Generative AI 
A Technical Introduction 
Generative artificial intelligence (GenAI) tools are an emerging class of new-age artificial 
intelligence algorithms capable of producing novel content — in varied formats such as text, 
audio, video, pictures, and code — based on user prompts. Recent advances in machine 
learning (ML), massive datasets, and substantial increases in computing power have propelled 
such tools to human-level performance on academic and professional benchmarks, 1
comparable to the ninetieth percentile on the SAT and the bar exam. 
This rapid progress has led many to to believe that the metamorphosis of these technologies 2
from research-grade demos to accessible and easy-to-use production-grade goods and 
services carries the potential to supercharge business processes and operations while enabling 
entirely new deliverables heretofore rendered infeasible by economic or technological factors. I

In [ ]:
##Split the pdf into chunks

In [22]:
text_spliter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_spliter.split_documents(doc)
print(chunks)

[Document(metadata={'producer': 'macOS Version 14.0 (Build 23A344) Quartz PDFContext', 'creator': 'Pages', 'creationdate': '2023-10-25T02:42:03+00:00', 'author': 'Karan Singh', 'moddate': '2023-10-25T09:40:35-04:00', 'title': 'GenAI-Principles', 'source': 'genai-principles.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Karan Singh, Assistant Professor of Operations Research \nPrinciples of Generative AI \nA Technical Introduction \nGenerative artificial intelligence (GenAI) tools are an emerging class of new-age artificial \nintelligence algorithms capable of producing novel content — in varied formats such as text, \naudio, video, pictures, and code — based on user prompts. Recent advances in machine \nlearning (ML), massive datasets, and substantial increases in computing power have propelled \nsuch tools to human-level performance on academic and professional benchmarks, 1\ncomparable to the ninetieth percentile on the SAT and the bar exam. \nThis rapid progre

In [23]:
print("Total Chunks:",len(chunks))

Total Chunks: 32


In [24]:
print(chunks[0].page_content)

Karan Singh, Assistant Professor of Operations Research 
Principles of Generative AI 
A Technical Introduction 
Generative artificial intelligence (GenAI) tools are an emerging class of new-age artificial 
intelligence algorithms capable of producing novel content — in varied formats such as text, 
audio, video, pictures, and code — based on user prompts. Recent advances in machine 
learning (ML), massive datasets, and substantial increases in computing power have propelled 
such tools to human-level performance on academic and professional benchmarks, 1
comparable to the ninetieth percentile on the SAT and the bar exam. 
This rapid progress has led many to to believe that the metamorphosis of these technologies 2
from research-grade demos to accessible and easy-to-use production-grade goods and 
services carries the potential to supercharge business processes and operations while enabling


In [ ]:
 ### Embeding

In [25]:
embedding=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
test_emb=embedding.embed_query("What is GenAi?")
print("Embeding Vector Size:",len(test_emb))
print("First 20  values:",test_emb[:20])

Embeding Vector Size: 384
First 20  values: [-0.05150924623012543, 0.08072686940431595, -0.046062760055065155, 0.032156869769096375, 0.013491127640008926, -0.034244511276483536, 0.06035760045051575, 0.018627284094691277, -0.005872226320207119, -0.06703125685453415, 0.09190259873867035, -0.0387280099093914, -0.012189559638500214, 0.013898271135985851, 0.02158348262310028, -0.01028226688504219, 0.04632379487156868, 0.021632421761751175, -0.006536665838211775, -0.021164115518331528]


In [ ]:
### Create the  FAISS Vector Store

In [27]:
vector_store=FAISS.from_documents(chunks,embedding)
print("Vector Store Successful!")

Vector Store Successful!


In [28]:
retriver=vector_store.as_retriever(search_kwargs={"k":4})

In [29]:
query=input("Enter Your Questions:")
retriver_doc=retriver.invoke(query)

Enter Your Questions:genai


In [30]:

for i, doc in enumerate(retriver_doc, start=1):
    print(f"\n--- Retrieved Document {i} ---")
    print(doc.page_content)
    print("Page:", doc.metadata.get("page", "unknown"))


--- Retrieved Document 1 ---
product deployment, for example, by Adobe for creating visual content and by Github as a 
programming assistance tool.   
 2
Figure 2: An image-
based GenAI model, 
Midjourney’s response to 
the prompt — 
“Businessman in Tokyo 
amidst rush hour, his 
gaze fixed ahead, 
surrounded by a sea of 
black umbrellas.”
Figure 3: Based on a code-based GenAI model, OpenAI Codex, 
Github Copilot is a commercial tool that can generate functional 
code from specifications given as natural language. Reportedly, as 
of June 2023, it served over a million users.
Page: 1

--- Retrieved Document 2 ---
Karan Singh, Assistant Professor of Operations Research 
Purpose and Scope  
What are these new-era AI technologies? How do they function? What principles do they 
operate on? What makes them different than already-hyped-up conventional machine learning 
(ML) models? For what tasks is this class of technology most impactful? What future advances 
might one look forward to? Thes

In [ ]:
### Configure the llm

In [31]:
os.environ["GROQ_API_KEY"]="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"
llm=ChatGroq(
model="openai/gpt-oss-120b",
temperature=0.2
)
print("LLM Ready!")

LLM Ready!


In [32]:
def format_docs(docs):
    return "\n\n".join(
        f"[Page {doc.metadata.get('page', 0) + 1}]\n{doc.page_content}"
        for doc in docs
    )

context = format_docs(retriver_doc)
print(context)

[Page 2]
product deployment, for example, by Adobe for creating visual content and by Github as a 
programming assistance tool.   
 2
Figure 2: An image-
based GenAI model, 
Midjourney’s response to 
the prompt — 
“Businessman in Tokyo 
amidst rush hour, his 
gaze fixed ahead, 
surrounded by a sea of 
black umbrellas.”
Figure 3: Based on a code-based GenAI model, OpenAI Codex, 
Github Copilot is a commercial tool that can generate functional 
code from specifications given as natural language. Reportedly, as 
of June 2023, it served over a million users.

[Page 2]
Karan Singh, Assistant Professor of Operations Research 
Purpose and Scope  
What are these new-era AI technologies? How do they function? What principles do they 
operate on? What makes them different than already-hyped-up conventional machine learning 
(ML) models? For what tasks is this class of technology most impactful? What future advances 
might one look forward to? These are the questions this report attempts to shed 

In [33]:

prompt = ChatPromptTemplate.from_template(f'''
You are the official AI assistant for Govardhan Institute and Infotech.

Answer the user's question using ONLY the information in the PDF context.

Context:
{{context}}

User Question:
{{question}}

Rules:
1. Use only the provided context.
2. Do not invent or assume information.
3. Answer clearly when the context supports the answer.
4. If the context does not contain enough information, do not guess.


Answer: ''')

In [34]:
question = input("Enter the  Questions:")
docs = retriver.invoke(question)
context = format_docs(docs)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(final_prompt.to_string())

Enter the  Questions:genai
Human: 
You are the official AI assistant for Govardhan Institute and Infotech.

Answer the user's question using ONLY the information in the PDF context.

Context:
[Page 2]
product deployment, for example, by Adobe for creating visual content and by Github as a 
programming assistance tool.   
 2
Figure 2: An image-
based GenAI model, 
Midjourney’s response to 
the prompt — 
“Businessman in Tokyo 
amidst rush hour, his 
gaze fixed ahead, 
surrounded by a sea of 
black umbrellas.”
Figure 3: Based on a code-based GenAI model, OpenAI Codex, 
Github Copilot is a commercial tool that can generate functional 
code from specifications given as natural language. Reportedly, as 
of June 2023, it served over a million users.

[Page 2]
Karan Singh, Assistant Professor of Operations Research 
Purpose and Scope  
What are these new-era AI technologies? How do they function? What principles do they 
operate on? What makes them different than already-hyped-up conventional 

In [35]:
response=llm.invoke(final_prompt)
print(response.content)

**Generative AI (GenAI)** refers to a newer class of artificial‑intelligence algorithms that can create novel content—such as text, audio, video, pictures, and code—when given user prompts.  

Key points from the report:

- **Capability** – GenAI tools generate new material in many formats (text, audio, video, images, code) based on the instructions they receive.  
- **Recent advances** – Improvements in machine‑learning techniques, the availability of massive datasets, and much greater computing power have pushed these tools to human‑level performance on academic and professional benchmarks (e.g., scoring in the 90th percentile on the SAT and the bar exam).  
- **Focus of the report** – Although generative AI models exist for images, code, audio, etc., the report concentrates on **large language models (LLMs)** because they dominate overall usage and because language is central to many information‑processing tasks.  
- **Commercial deployments** – Examples of successful product use in

In [ ]:
#### RAG CHAIN

In [40]:
rag_chain=(
    {
        "context":retriver | format_docs,
        "question":RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
)
print("Rag Chain Create Successfuly!")

Rag Chain Create Successfuly!


In [41]:

question = "What is  the purpose and scope of genai?"
answer = rag_chain.invoke(question)
print("Question:", question)
print("Answer:", answer)



Question: What is  the purpose and scope of genai?
Answer: **Purpose**  
The report aims to answer the fundamental questions about new‑era AI technologies (GenAI): what they are, how they work, the principles they operate on, how they differ from conventional machine learning models, which tasks they impact most, and what future advances may look like. It also seeks to “tease out how this understanding foundationally informs the best uses (and misuses) of GenAI in applied contexts.” 【Page 2 – Purpose and Scope】

**Scope**  
The discussion is centered on generative‑AI models, especially large language models (LLMs) that generate novel text from prompts, because LLMs drive most of the usage of GenAI and language is central to many information‑processing tasks. While the focus is on LLMs, the report acknowledges that image‑ and code‑based GenAI models (e.g., Adobe’s visual‑content tools and GitHub’s programming‑assistant tools) have also seen commercial deployment. 【Page 2 – “While genera

In [42]:
while True:
    question = input("\nAsk about this pdf : ")

    if question.strip().lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    try:
        answer = rag_chain.invoke(question)
        print("\nAI:", answer)
    except Exception as e:
        print("\nError:", e)


Ask about this pdf : what is the scope and purpose  of genai

AI: **Purpose**  
The report aims to answer fundamental questions about new‑era AI technologies—how they work, the principles behind them, how they differ from conventional machine‑learning models, the tasks where they are most impactful, and what future advances may look like. It also seeks to show how this understanding can guide the **best uses (and avoid misuses) of Generative AI (GenAI) in applied contexts**【Page 2】.

**Scope**  
While GenAI includes many model types (text, image, code, audio, video), the report concentrates **almost exclusively on large language models (LLMs) that generate novel text from textual prompts**, because LLMs currently drive the majority of GenAI usage and language is central to most information‑processing tasks. The discussion also acknowledges successful commercial deployments of image‑ and code‑based GenAI (e.g., Adobe’s visual‑content tools and GitHub Copilot)【Page 2】.

Ask about this p